<a href="https://colab.research.google.com/github/KarAnalytics/code_demos/blob/main/AutonomousAgent_BusinessValidator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. INSTALL DEPENDENCIES
!pip install -q git+https://github.com/KarAnalytics/llm_cascade.git


  Preparing metadata (setup.py) ... done


**IGNORE THE ABOVE ERRORS AND PROCEED** — they are specific to Google Colab's preinstalled packages and the code will still execute correctly.


# Autonomous Agent: Business Idea Validator

This notebook demonstrates an **autonomous workflow** — an agent that takes a high-level goal and figures out the steps itself, without a predefined pipeline or human guidance at each step.

**The goal:** Given a one-sentence business idea, produce a validation report with a go/no-go recommendation.

**The agent decides on its own:**
- What research questions to ask (planning)
- How to answer each one (execution)
- How to synthesize findings into a recommendation
- Whether the recommendation is solid or needs improvement (self-critique)

### How is this different from the other agent notebooks?

| | Tool-calling agent (LangChain) | Multi-agent system (MultiAgent_DB) | **Autonomous workflow (this)** |
|---|---|---|---|
| Who decides the task steps? | Agent picks tools at each step | Humans pre-define the pipeline | **Agent plans the whole workflow** |
| Fixed or dynamic pipeline? | Dynamic (tool-by-tool) | Fixed (Agent 1 → 2 → 3...) | **Dynamic (plan changes per input)** |
| Self-critique? | No | Only if an Evaluator agent is added | **Yes, as a built-in phase** |
| Typical use case | Answer questions using external tools | Well-defined workflows (ETL, BI) | **Open-ended goals (research, writing, validation)** |

**Why it matters:** Autonomous workflows are what people usually mean when they say "AI agent" in the wild — systems that take a goal and figure out how to achieve it. This notebook shows the simplest possible version: **Plan → Execute → Synthesize → Reflect**.


## Imports and Setup

Minimal dependencies: just `llm_cascade` for automatic LLM provider fallback. No tools, no databases, no vector stores — this is a pure LLM reasoning workflow.


In [2]:
from llm_cascade import get_cascade

llm = get_cascade()


LLM Cascade - available providers:
  + Gemini           model=gemini-2.5-flash
  + Ollama           model=kimi-k2.5:cloud
  + Groq             model=llama-3.3-70b-versatile
  + HuggingFace      model=meta-llama/Llama-3.3-70B-Instruct
  + Cohere           model=command-a-03-2025
  + OpenRouter       model=meta-llama/llama-3.3-70b-instruct:free
  + OpenAI           model=gpt-4o-mini
Not configured (skipped):
  - Grok (xAI)       (set XAI_API_KEY)


## The Four Phases of the Autonomous Workflow

Each phase is a single function that makes one LLM call with a phase-specific system prompt. The four phases together form the autonomous loop:

1. **Plan** — Given the goal, generate 4–5 specific questions that need to be answered
2. **Execute** — Loop through the plan and answer each question, passing prior answers as context
3. **Synthesize** — Combine all answers into a final recommendation
4. **Reflect** — Critique the recommendation and assign a confidence score

Notice that **each phase has a different system prompt** — not because we want multiple agent "personas" (like in `MultiAgent_DB`), but because each phase has a different cognitive task. The planner needs to think strategically; the executor needs to think concretely; the synthesizer needs to weigh tradeoffs; the critic needs to be skeptical.


In [3]:
# Phase 1: Planner -- decides WHAT needs to be researched
PLANNER_SYSTEM = (
    'You are a strategic business planner. Given a high-level goal, break it down '
    'into the 4-5 most critical questions that must be answered to achieve it. '
    'Output ONLY a numbered list of questions. No preamble, no conclusion.'
)


def plan(goal):
    '''Phase 1: generate a research plan as a list of questions.'''
    prompt = f'Goal: {goal}' + chr(10) + chr(10) + 'List the 4-5 most critical questions to answer.'
    response = llm.generate(prompt, system_prompt=PLANNER_SYSTEM)

    # Parse the numbered list into a Python list of strings
    steps = []
    for line in response.text.split(chr(10)):
        line = line.strip()
        if not line:
            continue
        # Strip leading '1.', '1)', '-', '*', etc.
        cleaned = line.lstrip('0123456789.)- *').strip()
        if cleaned and len(cleaned) > 5:
            steps.append(cleaned)
    return steps


print('Phase 1 (Planner) ready.')


Phase 1 (Planner) ready.


In [4]:
# Phase 2: Executor -- answers ONE research question at a time
EXECUTOR_SYSTEM = (
    'You are a business analyst. Answer the question concisely (2-4 sentences) '
    'with concrete reasoning. Use the prior context if helpful, but stay focused '
    'on the current question.'
)


def execute_step(question, prior_context):
    '''Phase 2: answer one research question with access to prior answers.'''
    prompt = 'Prior context:' + chr(10) + prior_context + chr(10) + chr(10) + 'Question: ' + question
    response = llm.generate(prompt, system_prompt=EXECUTOR_SYSTEM)
    return response.text


print('Phase 2 (Executor) ready.')


Phase 2 (Executor) ready.


In [5]:
# Phase 3: Synthesizer -- combines findings into a recommendation
SYNTHESIZER_SYSTEM = (
    'You are a senior executive. Given research findings, synthesize them into a '
    'clear, decisive recommendation. State the recommendation (go / no-go / conditional), '
    'followed by 3 bullet points of key reasoning.'
)


def synthesize(goal, research_results):
    '''Phase 3: combine all research answers into a final recommendation.'''
    findings = chr(10).join([f'Q: {q}' + chr(10) + f'A: {a}' for q, a in research_results])
    prompt = (
        'Goal: ' + goal + chr(10) + chr(10)
        + 'Research findings:' + chr(10) + findings + chr(10) + chr(10)
        + 'Synthesize into a final recommendation.'
    )
    response = llm.generate(prompt, system_prompt=SYNTHESIZER_SYSTEM)
    return response.text


print('Phase 3 (Synthesizer) ready.')


Phase 3 (Synthesizer) ready.


In [6]:
# Phase 4: Critic -- self-reflects on the recommendation
CRITIC_SYSTEM = (
    'You are a skeptical business critic. Given a recommendation, identify '
    'what is weak or missing, what is strong, and assign a confidence level '
    'from 1 (very weak) to 10 (very strong). Be concise.'
)


def reflect(recommendation):
    '''Phase 4: critique the recommendation and assign confidence.'''
    prompt = 'Recommendation to critique:' + chr(10) + recommendation
    response = llm.generate(prompt, system_prompt=CRITIC_SYSTEM)
    return response.text


print('Phase 4 (Critic) ready.')


Phase 4 (Critic) ready.


## The Autonomous Loop

This function ties everything together. Given just a one-sentence goal, it runs through all four phases **without any human input in the middle**. It prints progress as each phase completes, so students can watch the agent "think out loud."

Total LLM calls per run: **1 (plan) + 4–5 (execute) + 1 (synthesize) + 1 (reflect) = 7–8 calls**. Typical runtime on free-tier providers: ~1–2 minutes.


In [7]:
def run_autonomous_workflow(goal, verbose=True):
    '''The full autonomous workflow: Plan -> Execute -> Synthesize -> Reflect.'''
    if verbose:
        print('=' * 70)
        print('GOAL:', goal)
        print('=' * 70)

    # --- Phase 1: Plan ---
    if verbose:
        print(chr(10) + '[PHASE 1 - PLANNING]')
    steps = plan(goal)
    if verbose:
        print(f'Agent decided to research {len(steps)} questions:')
        for i, s in enumerate(steps, 1):
            print(f'  {i}. {s}')

    # --- Phase 2: Execute each step ---
    if verbose:
        print(chr(10) + '[PHASE 2 - EXECUTION]')
    results = []
    prior_context = ''
    for i, step in enumerate(steps, 1):
        if verbose:
            print(chr(10) + f'Step {i}: {step}')
        answer = execute_step(step, prior_context)
        results.append((step, answer))
        prior_context += chr(10) + f'Q: {step}' + chr(10) + f'A: {answer}'
        if verbose:
            print(f'  -> {answer}')

    # --- Phase 3: Synthesize ---
    if verbose:
        print(chr(10) + '[PHASE 3 - SYNTHESIS]')
    recommendation = synthesize(goal, results)
    if verbose:
        print(recommendation)

    # --- Phase 4: Reflect ---
    if verbose:
        print(chr(10) + '[PHASE 4 - SELF-CRITIQUE]')
    critique = reflect(recommendation)
    if verbose:
        print(critique)

    return {
        'goal': goal,
        'plan': steps,
        'research': results,
        'recommendation': recommendation,
        'critique': critique,
    }


print('Autonomous workflow function ready.')


Autonomous workflow function ready.


## Run the Agent

Edit the `business_idea` string below and run the cell. The agent will autonomously plan its research, execute each step, synthesize a recommendation, and self-critique — with no further input from you.

Try different ideas to see how the plan changes. The agent generates a different research plan for a fintech startup vs. a coffee shop vs. a SaaS tool.


In [8]:
business_idea = 'An AI-powered app that helps college students find affordable off-campus housing by predicting rent trends.'

# business_idea = 'A subscription meal kit service for busy professionals that uses local ingredients.'
# business_idea = 'A mobile game that teaches kids basic accounting through story-based puzzles.'

result = run_autonomous_workflow(business_idea)


GOAL: An AI-powered app that helps college students find affordable off-campus housing by predicting rent trends.

[PHASE 1 - PLANNING]
  [Response from Gemini / gemini-2.5-flash]
Agent decided to research 5 questions:
  1. What are the key features and data sources required to accurately predict rent trends in various college towns?
  2. How will we acquire, verify, and continuously update real-time housing availability and pricing data from diverse sources (e.g., listings, historical data, local market reports)?
  3. What is the most effective go-to-market strategy to reach and acquire college students, build trust, and drive app adoption?
  4. What is the sustainable business model that balances affordability for students with profitability for the app, and what are the key revenue streams?
  5. How will we ensure the AI's predictions are accurate, unbiased, and adaptable to rapidly changing market conditions and individual student preferences?

[PHASE 2 - EXECUTION]

Step 1: What a

## Inspect the Full Trace

The `result` dict holds everything the agent did, so you can examine each phase individually. This is useful for debugging or for teaching how the agent's reasoning flowed from plan to final recommendation.


In [9]:
print('=' * 70)
print('FULL TRACE')
print('=' * 70)
print(chr(10) + 'GOAL:', result['goal'])
print(chr(10) + 'PLAN (what the agent decided to research):')
for i, step in enumerate(result['plan'], 1):
    print(f'  {i}. {step}')
print(chr(10) + 'FINAL RECOMMENDATION:')
print(result['recommendation'])
print(chr(10) + 'SELF-CRITIQUE:')
print(result['critique'])


FULL TRACE

GOAL: An AI-powered app that helps college students find affordable off-campus housing by predicting rent trends.

PLAN (what the agent decided to research):
  1. What are the key features and data sources required to accurately predict rent trends in various college towns?
  2. How will we acquire, verify, and continuously update real-time housing availability and pricing data from diverse sources (e.g., listings, historical data, local market reports)?
  3. What is the most effective go-to-market strategy to reach and acquire college students, build trust, and drive app adoption?
  4. What is the sustainable business model that balances affordability for students with profitability for the app, and what are the key revenue streams?
  5. How will we ensure the AI's predictions are accurate, unbiased, and adaptable to rapidly changing market conditions and individual student preferences?

FINAL RECOMMENDATION:
## Recommendation: GO

This project presents a compelling opport

## Key Takeaways

**What makes this workflow "autonomous":**
- You provide only a **high-level goal**, not a list of steps
- The agent generates its own **research plan** based on the goal
- Each step is executed with access to the **prior context** (results accumulate)
- The agent self-critiques its output, giving you a **confidence signal** for free

**What's NOT happening (and why this example is simple):**
- No external tools (no web search, no database queries)
- No iteration: the agent doesn't re-plan based on the critique
- No parallelism: execution is strictly sequential
- No human-in-the-loop: the workflow runs end-to-end without interruption

**Scaling this up in the real world:**
- Add **tools** so execution steps can fetch real data (web search, database, APIs)
- Add an **iteration loop**: if the critic's confidence is low, feed the critique back to the planner to generate a better plan
- Use **LangGraph** to express this as a stateful graph with conditional edges (see `LangGraph_demo.ipynb` for the foundation)
- Add **memory** so the agent can remember prior runs and improve over time

**Comparison with our other notebooks:**

| Notebook | Who decides the steps? |
|---|---|
| `LlamaIndex_RAG`, `LangChain_demo` (RAG chain) | Humans — the pipeline is fixed |
| `LangChain_demo` (agent with tools) | LLM picks tools at each turn, but the goal is one question |
| `MultiAgent_DB` | Humans — 5 specialized agents run in a fixed order |
| `SingleAgent_DB` | Humans — one agent runs a fixed workflow |
| **This notebook** | **LLM decides the whole workflow from a one-sentence goal** |

**Exercises:**
- Try a business idea the LLM is unlikely to have strong prior knowledge about (e.g., a niche B2B tool). Does the plan still look reasonable?
- Modify `run_autonomous_workflow` to loop: after the critic phase, if confidence < 7, re-run the plan with the critique appended to the goal.
- Add a fifth phase that outputs a one-page marketing pitch based on the recommendation.
- Replace the `PLANNER_SYSTEM` prompt with a different persona (e.g., "venture capitalist") and see how the plan changes.
